# 08D_Final_v2_Skill_Taxonomy_Builder

Rule-first skill classification framework for HSEP.

Output:
- expanded_skill_master.csv

Columns:
- skill
- linkedin_frequency
- category
- sub_category
- confidence


In [1]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
skills = pd.read_csv('../Generated Datasets/linkedin_skill_universe_1000.csv')
skills['skill'] = skills['skill'].astype(str).str.lower().str.strip()

## Technical Taxonomy

In [3]:
TECHNICAL = {
    'Programming': ['python','java','javascript','typescript','cpp','csharp','r','go','rust'],
    'Database': ['sql','mysql','postgresql','mongodb','oracle','nosql','snowflake'],
    'Cloud': ['aws','azure','gcp','google cloud','cloud computing'],
    'DevOps': ['docker','kubernetes','jenkins','gitlab','github actions','terraform'],
    'AI_ML': ['machine learning','deep learning','computer vision','natural language processing','llm','rag','langchain','tensorflow','pytorch'],
    'Data Analytics': ['tableau','power bi','powerbi','excel','data analysis','analytics','statistics'],
    'Cybersecurity': ['cybersecurity','network security','cloud security','penetration testing'],
    'Software Engineering': ['agile development','software development','software engineering','oop'],
    'Data Engineering': ['data engineering','etl','data warehouse','big data']
}

## Soft Skill Taxonomy

In [4]:
SOFT = {
    'Communication':['communication','communication skills','written communication','verbal communication'],
    'Leadership':['leadership','people management'],
    'Collaboration':['teamwork','collaboration','relationship building'],
    'Problem Solving':['problem solving','critical thinking','analytical thinking'],
    'Personal Effectiveness':['adaptability','time management','organization','multitasking']
}

## Domain Knowledge Taxonomy

In [5]:
DOMAIN = {
    'Healthcare':['patient care','nursing','medical imaging','family medicine','medical oncology','clinical psychology'],
    'Finance':['accounting','finance','financial analysis','budgeting'],
    'Sales':['sales','cold calling','sales presentations'],
    'Marketing':['marketing','market analysis','social media marketing'],
    'HR':['recruitment','human resources','talent acquisition'],
    'Education':['teaching','education','curriculum development'],
    'Legal':['corporate law','legal services'],
    'Operations':['operations','inventory management','supply chain','logistics'],
    'Construction':['construction','carpentry'],
    'Manufacturing':['manufacturing','quality control']
}

In [6]:
CERT_PATTERNS = [r'.*certification.*', r'.*certified.*', r'.*license.*', r'.*credential.*']

LANGUAGE_TERMS = ['english','spanish','french','german','hindi','arabic','japanese','chinese']

NOISE_PATTERNS = [
    r'.*insurance.*',
    r'.*benefits.*',
    r'.*401k.*',
    r'.*retirement.*',
    r'.*stipend.*',
    r'.*paid time off.*'
]

## NLP Fallback

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

labels = {
    'Technical':'technology software programming cloud ai database analytics',
    'Soft Skill':'communication teamwork leadership interpersonal',
    'Domain Knowledge':'healthcare finance sales marketing hr education legal operations',
    'Certification':'certification credential license',
    'Language':'english spanish french hindi',
    'Noise':'benefits insurance retirement stipend'
}

label_names = list(labels.keys())
label_embeddings = model.encode(list(labels.values()), normalize_embeddings=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9751.11it/s]


## Classification Engine

In [8]:
results = []

for skill in skills['skill']:
    classified = False

    for group_name, group in [('Technical',TECHNICAL),('Soft Skill',SOFT),('Domain Knowledge',DOMAIN)]:
        for subcat, values in group.items():
            if skill in values:
                results.append([group_name, subcat, 1.0])
                classified = True
                break
        if classified:
            break

    if classified:
        continue

    if any(re.match(p, skill) for p in CERT_PATTERNS):
        results.append(['Certification','Certification',0.95])
        continue

    if skill in LANGUAGE_TERMS:
        results.append(['Language','Language',0.95])
        continue

    if any(re.match(p, skill) for p in NOISE_PATTERNS):
        results.append(['Noise','Noise',0.95])
        continue

    all_terms = []
    for grp in [TECHNICAL,SOFT,DOMAIN]:
        for vals in grp.values():
            all_terms.extend(vals)

    fuzzy_match = process.extractOne(skill, all_terms, scorer=fuzz.token_sort_ratio)

    if fuzzy_match and fuzzy_match[1] >= 90:
        results.append(['Review','Fuzzy_Match',0.85])
        continue

    emb = model.encode([skill], normalize_embeddings=True)
    sims = cosine_similarity(emb, label_embeddings)[0]
    idx = sims.argmax()
    conf = float(sims[idx])

    if conf >= 0.70:
        results.append([label_names[idx], label_names[idx], conf])
    else:
        results.append(['Review','Review',conf])

In [9]:
skills[['category','sub_category','confidence']] = pd.DataFrame(results)

cols = ['skill']

freq_col = None
for c in skills.columns:
    if c not in ['skill','category','sub_category','confidence']:
        freq_col = c
        break

if freq_col:
    cols.append(freq_col)

final_df = skills[cols + ['category','sub_category','confidence']]

if freq_col:
    final_df.columns = ['skill','linkedin_frequency','category','sub_category','confidence']

final_df.to_csv('../Generated Datasets/expanded_skill_master.csv', index=False)

print(final_df['category'].value_counts())

category
Review              2402
Noise                 60
Certification         58
Technical             45
Domain Knowledge      32
Soft Skill            17
Language               3
Name: count, dtype: int64
